In [5]:
import pandas as pd
csv_file_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/Kinmen_20241205.csv"
df = pd.read_csv(csv_file_path)
df.shape

(668, 11)

# BERTopic model訓練一
如果要訓練split by period 的資料，我用筆電跑會有點久（跑了20分鐘都沒有訓練完成）

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re

# 定義噪音字元集合
noise_chars = {'💎', '。', '▲', '△', '🔍', '？', '—', '<', '∶', '\r', '；', '✦', '\u200c', '️', '－', '℃', '‖', '!', '「', '→', '/', '║', '」', '@', '，', '?', "'", '○', '）', '『', '．', '👉', '】', '🌟', '=', '👇', '‰', '【', ';', '#', ')', '：', '\u200d', '❖', '~', ']', '%', '·', '↑', '（', '〕', '☆', '※', '&', '•', '👍', '>', '／', '▌', '–', '↓', '[', '’', ':', '《', '▎', '🤝', '©', '+', '🌊', '\xa0', '\n', '◇', ',', '◎', '…', '(', '〔', '\\', '“', '■', '｜', '─', '\u200b', '-', '●', '"', '▊', '、', '︱', '‘', '*', '⭐', '》', '％', '！', '〉', '|', '▼', '👆', '🏡', '°', '\t', '』', '〈', '～', '◆', '.', '⬆', '”'}

# 定義清理函數
def clean_text(text, noise_chars):
    # 使用正則表達式移除噪音字元
    noise_pattern = f"[{''.join(re.escape(char) for char in noise_chars)}]"
    return re.sub(noise_pattern, "", text)

# 步驟 0: 讀取 CSV 文件
csv_file_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/Kinmen_20241205_split_by_period.csv"
df = pd.read_csv(csv_file_path)

# 確保 content 欄位存在
if 'content' not in df.columns:
    raise ValueError("The CSV file does not contain a 'content' column.")

# 提取文本數據並清理噪音字元
df['cleaned_content'] = df['content'].dropna().apply(lambda x: clean_text(x, noise_chars))
texts = df['cleaned_content'].tolist()

# 步驟 1: 嵌入向量生成
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # 輕量級嵌入模型
embeddings = embedding_model.encode(texts)

# 步驟 2: 降維處理
umap_model = UMAP(n_neighbors=15, n_components=2, metric='cosine')

# 步驟 3: 主題聚類
hdbscan_model = HDBSCAN(min_cluster_size=2, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# 步驟 4: 讀取停用詞文件
stopwords_file_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/hit_stopwords.txt"
with open(stopwords_file_path, encoding='utf-8') as f:
    stop_words = [line.strip() for line in f]

# 配置 CountVectorizer
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words=stop_words)

# 步驟 5: 主題建模（啟用 N-gram 支援）
topic_model_split = BERTopic(
    embedding_model=embedding_model, 
    verbose=True,
    calculate_probabilities=True,
    nr_topics="auto",
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer
)

# 擬合模型並提取主題
topics, probs = topic_model_split.fit_transform(texts)

# 查看主題信息
print(topic_model_split.get_topic_info())

# 步驟 6: 可視化
topic_model_split.visualize_barchart(top_n_topics=5).show()

2024-12-16 15:17:13,387 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/615 [00:00<?, ?it/s]

2024-12-16 15:17:32,177 - BERTopic - Embedding - Completed ✓
2024-12-16 15:17:32,177 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2024-12-16 15:17:44,206 - BERTopic - Dimensionality - Completed ✓
2024-12-16 15:17:44,207 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the

In [3]:
#model saved
topic_model.save("/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/model/bertopic_model_version2")


2024-12-16 11:47:44,328 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [4]:
# 列出所有主題的資訊
topic_info = topic_model.get_topic_info()

# 印出主題類別
print(topic_info)


     Topic  Count                                               Name  \
0       -1    121  -1_回复_作者詹方歌来源豹变idbaobiannews豹变idbaobiannews江疏影...   
1        0     18  0_据新华社消息14日下午福建一艘渔船在金门海域被台方驱离导致船上4人全部落海其中2人死亡对...   
2        1     14  1_2024年春节后发生在厦金海域上的大陆渔民遇难事件已经超过十天可是还没有迎来台方的道歉顽...   
3        2     13  2_不过据我的观察大陆这边的反制措施可能不仅是军事方面而是寻求事实上的统一比如在厦金海域大陆...   
4        3     11  3_我们可以理解大陆暂时不想打仗的战略但在战争与妥协之间还有很多的选项在10几年前在我们还不...   
..     ...    ...                                                ...   
101    100      2  100_自214金门恶性撞船事件之后大陆海警船频繁进入金门所谓的禁止水域进行常态执法巡查甚至...   
102    101      2  101_事件的背后是两岸关系的微妙交织国台办发言人朱凤莲对管碧玲的态度表示不满指责其推诿责任...   
103    102      2  102_原野投稿邮箱cxt6901163com投稿微信cheng19690101相关说明古典...   
104    103      2  103_点击上方蓝字免费订阅本账号文 小兔扒八我欣然接受所有因为时间会证明一切在台海风云变幻...   
105    104      2  104_主编张志达编辑秦静_主编张志达编辑秦静 校对高少卓央视新闻央视新闻_自上月起2月14...   

                                        Representation  \
0    [回复, 作者詹方歌来源豹变idbaobiannews豹变idbaobiannews江疏影到...   
1    [据新华社消息14日下午福建

In [10]:
from bertopic import BERTopic
import pandas as pd

# 1. 載入已保存的模型
model_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/model/bertopic_model_version1"
topic_model = BERTopic.load(model_path)

# 2. 讀取新的文本數據
data_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/Kinmen_20241205.csv"
df = pd.read_csv(data_path)

# 確保 content 欄位非空
df = df.dropna(subset=["content"])

# 3. 提取文本進行主題分析
texts = df["content"].tolist()
print(df["content"].head())  # 查看文本內容
print(df["content"].isnull().sum())  # 確保沒有空值
topics, probs = topic_model.transform(texts)

# 4. 整合分析結果
# 添加主題和概率到原始數據
df["topic"] = topics
df["probability"] = probs

# 獲取主題名稱和關鍵詞
topic_info = topic_model.get_topic_info()
topic_mapping = topic_info.set_index("Topic")["Name"].to_dict()  # 主題 ID 對應名稱
df["topic_name"] = df["topic"].map(topic_mapping)

# 5. 保存整併後的結果
output_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/results/kinmen_topic_results.csv"
df.to_csv(output_path, index=False)

print(f"整併後的結果已保存至 {output_path}")


0    在两岸关系的敏感背景下，金门海域渔船事故成为了一触即发的导火索。台方的行动不仅造成了人员伤亡...
1    理解了，让我们扩展一下这篇文章，确保达到至少1200字以上的要求。大陆海警频繁执法金门等海域...
2    时政丨热点丨军事丨为农这里，是龙牙正在爬的一座山。据《联合早报》报道，台“海洋委员会主任委员...
3        2024年3月12日，算是开年打假反诈骗第二文。话说江苏陈五哥给我发来几个文件请我帮...
4    2024年6月12日，国务院台湾事务办公室举行例行新闻发布会，国台办发言人陈斌华主持本次新闻...
Name: content, dtype: object
0


AttributeError: No prediction data was generated

In [11]:
import pandas as pd
from collections import Counter
import re

# 讀取數據
data_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/Kinmen_20241205.csv"
df = pd.read_csv(data_path)

# 確保 content 欄位非空
df = df.dropna(subset=["content"])

# 將所有文本合併成一個大字符串
all_text = " ".join(df["content"].tolist())

# 定義正則表達式匹配非語言字符
noise_pattern = r"[^\w\s\u4e00-\u9fff]"  # 匹配非字母、數字、漢字、空白的字符

# 找出所有匹配的噪音字元
noise_chars = set(re.findall(noise_pattern, all_text))

# 顯式檢查不可見字符
invisible_chars = ["\xa0", "\n", "\t", "\r"]

# 將不可見字符加入結果
for char in invisible_chars:
    noise_chars.add(char)

# 最終的噪音字元集合
print("最終的噪音字元集合：", noise_chars)

最終的噪音字元集合： {'💎', '。', '▲', '△', '🔍', '？', '—', '<', '∶', '\r', '；', '✦', '\u200c', '️', '－', '℃', '‖', '!', '「', '→', '/', '║', '」', '@', '，', '?', "'", '○', '）', '『', '．', '👉', '】', '🌟', '=', '👇', '‰', '【', ';', '#', ')', '：', '\u200d', '❖', '~', ']', '%', '·', '↑', '（', '〕', '☆', '※', '&', '•', '👍', '>', '／', '▌', '–', '↓', '[', '’', ':', '《', '▎', '🤝', '©', '+', '🌊', '\xa0', '\n', '◇', ',', '◎', '…', '(', '〔', '\\', '“', '■', '｜', '─', '\u200b', '-', '●', '"', '▊', '、', '︱', '‘', '*', '⭐', '》', '％', '！', '〉', '|', '▼', '👆', '🏡', '°', '\t', '』', '〈', '～', '◆', '.', '⬆', '”'}


---
之前訓練的降維度模型沒有保存到，所以有新的資料丟進到模型中，會出現轉換問題。

In [1]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN, generate_prediction_data
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re

# 定義噪音字元集合
noise_chars = {'💎', '。', '▲', '△', '🔍', '？', '—', '<', '∶', '\r', '；', '✦', '\u200c', '️', '－', '℃', 
               '‖', '!', '「', '→', '/', '║', '」', '@', '，', '?', "'", '○', '）', '『', '．', '👉', '】', 
               '🌟', '=', '👇', '‰', '【', ';', '#', ')', '：', '\u200d', '❖', '~', ']', '%', '·', '↑', '（', 
               '〕', '☆', '※', '&', '•', '👍', '>', '／', '▌', '–', '↓', '[', '’', ':', '《', '▎', '🤝', 
               '©', '+', '🌊', '\xa0', '\n', '◇', ',', '◎', '…', '(', '〔', '\\', '“', '■', '｜', '─', 
               '\u200b', '-', '●', '"', '▊', '、', '︱', '‘', '*', '⭐', '》', '％', '！', '〉', '|', '▼', 
               '👆', '🏡', '°', '\t', '』', '〈', '～', '◆', '.', '⬆', '”'}

# 定義清理函數
def clean_text(text, noise_chars):
    # 使用正則表達式移除噪音字元
    noise_pattern = f"[{''.join(re.escape(char) for char in noise_chars)}]"
    return re.sub(noise_pattern, "", text)

# 步驟 0: 讀取 CSV 文件
csv_file_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/Kinmen_20241205.csv"
df = pd.read_csv(csv_file_path)

# 確保 content 欄位存在
if 'content' not in df.columns:
    raise ValueError("The CSV file does not contain a 'content' column.")

# 提取文本數據並清理噪音字元
df['cleaned_content'] = df['content'].dropna().apply(lambda x: clean_text(x, noise_chars))
texts = df['cleaned_content'].tolist()

# 步驟 1: 嵌入向量生成
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # 輕量級嵌入模型
embeddings = embedding_model.encode(texts)

# 步驟 2: 降維處理
umap_model = UMAP(n_neighbors=15, n_components=2, metric='cosine')

# 步驟 3: 主題聚類（啟用 prediction_data）
hdbscan_model = HDBSCAN(min_cluster_size=2, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# 步驟 4: 讀取停用詞文件
stopwords_file_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/data/hit_stopwords.txt"
with open(stopwords_file_path, encoding='utf-8') as f:
    stop_words = [line.strip() for line in f]

# 配置 CountVectorizer
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words=stop_words)

# 步驟 5: 主題建模（啟用 N-gram 支援）
topic_model = BERTopic(
    embedding_model=embedding_model, 
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer
)

# 擬合模型並提取主題
topics, probs = topic_model.fit_transform(texts)

# 查看主題信息
print(topic_model.get_topic_info())

# 保存模型
model_save_path = "/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/model/bertopic_model_version3"
topic_model.save(model_save_path)
print(f"模型已保存至 {model_save_path}")

# 步驟 6: 可視化
topic_model.visualize_barchart(top_n_topics=5).show()


ImportError: cannot import name 'generate_prediction_data' from 'hdbscan' (/Users/shuyuhsu/miniconda3/envs/NLP/lib/python3.11/site-packages/hdbscan/__init__.py)

## 仿照BERTopic_embeddings.ipynb進行分析

In [3]:
%pip install paddlenlp bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas gensim

Note: you may need to restart the kernel to use updated packages.


In [3]:
from paddlenlp import Taskflow
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re
from gensim.models.phrases import Phrases, Phraser


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [2]:
%pip install paddlepaddle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 MB 4.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 1.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


### 新版本BERTopic model

In [2]:

# 定義噪音字元集合
noise_chars = {'💎', '。', '▲', '△', '🔍', '？', '—', '<', '∶', '\r', '；', '✦', '\u200c', '️', '－', '℃', '‖', '!', '「', '→', '/', '║', '」', '@', '，', '?', "'", '○', '）', '『', '．', '👉', '】', '🌟', '=', '👇', '‰', '【', ';', '#', ')', '：', '\u200d', '❖', '~', ']', '%', '·', '↑', '（', '〕', '☆', '※', '&', '•', '👍', '>', '／', '▌', '–', '↓', '[', '’', ':', '《', '▎', '🤝', '©', '+', '🌊', '\xa0', '\n', '◇', ',', '◎', '…', '(', '〔', '\\', '“', '■', '｜', '─', '\u200b', '-', '●', '"', '▊', '、', '︱', '‘', '*', '⭐', '》', '％', '！', '〉', '|', '▼', '👆', '🏡', '°', '\t', '』', '〈', '～', '◆', '.', '⬆', '”'}

# 定義清理函數
def clean_text(text, noise_chars):
    # 使用正則表達式移除噪音字元
    noise_pattern = f"[{''.join(re.escape(char) for char in noise_chars)}]"
    return re.sub(noise_pattern, "", text)

# 步驟 0: 讀取 CSV 文件
csv_file_path = "./data/Kinmen_20241205_split_by_period.csv"
df = pd.read_csv(csv_file_path)
print('empty string: ',(df["content"] == "").sum())
print(df.iloc[6089,:])

empty string:  0
_id                                               66765f9ffb88ab4b048dd65f
URL                      https://mp.weixin.qq.com/s/ffHEMK0wj0qpmSTLJBqqtQ
title                                       管碧玲道歉话音刚落，大陆渔民金门再遇难！我方要求登岛搜救遭拒
date                                                            2024-03-16
content                                                                  🌟
Official media                                                           0
author                                                                 NaN
media                                                                    1
account                                                                  1
Internet news license                                                    1
Commercial media                                                         1
chunk_id                                       66765f9ffb88ab4b048dd65f_16
original_id                                       66765f9ffb88ab4b048dd65f
chunk_si

In [3]:

# 確保 content 欄位存在
if 'content' not in df.columns:
    raise ValueError("The CSV file does not contain a 'content' column.")

# 提取文本數據並清理噪音字元
df['cleaned_content'] = df['content'].dropna().apply(lambda x: clean_text(x, noise_chars))
# 找到 cleaned_content 為空字符串的行
empty_rows = df[df["cleaned_content"] == ""]

# 列印這些行的內容
print("Rows with empty cleaned_content:")
print(empty_rows[["content", "cleaned_content"]])  # 可以選擇列印相關列

# # 檢查是否有 NaN 值
# print(df["cleaned_content"].isna().sum())
# # 檢查是否有空字符串
# print((df["cleaned_content"] == "").sum())
# # 檢查非字符串數據類型
# print(df["cleaned_content"].apply(lambda x: isinstance(x, str)).value_counts())

# 刪除這幾列
df = df.drop(empty_rows.index)
empty_rows = df[df["cleaned_content"] == ""]

# 列印這些行的內容
print("2 Rows with empty cleaned_content:")
print(empty_rows[["content", "cleaned_content"]])  # 可以選擇列印相關列

Rows with empty cleaned_content:
     content cleaned_content
2704       ”                
6089       🌟                
6464      ‍‍                
7605       。                
2 Rows with empty cleaned_content:
Empty DataFrame
Columns: [content, cleaned_content]
Index: []


In [4]:

# # 使用 PaddleNLP 的分詞工具進一步處理
# lac = Taskflow("word_segmentation")
# stopwords = set(["的", "在", "了", "是", "和", "也", "有", "就", "不", "对", "这", "上", "与", "及", "以", "为", "但", "而", "或", "等"])  # 自定義停用詞

# def preprocess_text(text):
#     tokens = lac(text)  # 使用 PaddleNLP 分詞
#     tokens = [token for token in tokens if token not in stopwords and token.strip()]  # 去除停用詞
#     return tokens

# df["tokens"] = df["cleaned_content"].apply(preprocess_text)

E1231 12:00:35.237040 4123247168 analysis_config.cc:658] Please compile with MKLDNN first to use MKLDNN


In [4]:
import jieba
import pandas as pd

# Step 1: 加載自定義字典
jieba.load_userdict("/Users/shuyuhsu/code_workspace/Xiemen_wechat_BERTTopic/jieba.dict.utf8.txt")  # 你的自定義字典檔案路徑

# Step 2: 定義停用詞
stopwords = set(["的", "在", "了", "是", "和", "也", "有", "就", "不", "对", "这", "上", "与", "及", "以", "为", "但", "而", "或", "等"])  # 自定義停用詞

# Step 3: 定義分詞和預處理函數
def preprocess_text(text):
    # 使用 jieba 進行分詞
    tokens = jieba.lcut(text)  # lcut 返回分詞後的詞語列表
    # 過濾停用詞和空白字符
    tokens = [token for token in tokens if token not in stopwords and token.strip()]
    return tokens

# Step 4: 將分詞應用到 DataFrame
# 假設你的 DataFrame 名為 df，且包含 `cleaned_content` 欄位
df["tokens"] = df["cleaned_content"].apply(preprocess_text)


# print(f"已完成分詞處理，結果儲存至: {output_path}")


Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/kd/zk4bd2dj0hx2g09y37627_lr0000gn/T/jieba.cache
Loading model cost 0.362 seconds.
Prefix dict has been built successfully.


已完成分詞處理，結果儲存至: processed_data_with_tokens.csv


In [5]:
# 使用 Gensim Phraser 進行 Bigram 識別
phrases = Phrases(df["tokens"], min_count=1, threshold=5)
bigram = Phraser(phrases)
df["tokens_bigram"] = df["tokens"].apply(lambda x: bigram[x])

# 將 Token 轉換回文本格式
df["processed_text"] = df["tokens_bigram"].apply(lambda x: " ".join(x))
texts = df["processed_text"].tolist()

In [6]:
from bertopic.vectorizers import ClassTfidfTransformer

# 步驟 1: 嵌入向量生成
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # 輕量級嵌入模型
embeddings = embedding_model.encode(texts)

# 步驟 2: 降維處理
umap_model = UMAP(n_neighbors=10, n_components=2, metric='cosine')

# 步驟 3: 主題聚類
hdbscan_model = HDBSCAN(min_cluster_size=100, metric='euclidean', cluster_selection_method='eom', prediction_data=True,cluster_selection_epsilon=0.1)

# 步驟 4: 讀取停用詞文件
stopwords_file_path = "./data/stop_words.txt"
with open(stopwords_file_path, encoding='utf-8') as f:
    stop_words = [line.strip() for line in f]

# 配置 CountVectorizer
vectorizer = CountVectorizer(ngram_range=(1, 3), stop_words=stop_words, max_features=10000)


# 配置 ClassTfidfTransformer 並添加 seed_words
ctfidf_model = ClassTfidfTransformer(
    seed_words=["兩岸", "漁權", "爭議", "海巡"],  # 你的 seed words
    bm25_weighting=True, 
    reduce_frequent_words=True
)

In [7]:
# 步驟 5: 主題建模（啟用 N-gram 支援）
topic_model = BERTopic(
    embedding_model=embedding_model, 
    verbose=True,
    calculate_probabilities=True,
    nr_topics="auto",
    umap_model=umap_model, 
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=15,
    min_topic_size=100,
    ctfidf_model=ctfidf_model # 添加 ClassTfidfTransformer
)

# 擬合模型並提取主題
topics, probs = topic_model.fit_transform(texts)

# 查看主題信息
print(topic_model.get_topic_info())
# 儲存模型到指定路徑
model_save_path = "./model/bertopic_seed_words_2D_newStopWords"
topic_model.save(model_save_path)

print(f"模型已儲存至 {model_save_path}")

2024-12-31 12:15:02,505 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 615/615 [00:20<00:00, 29.39it/s]
2024-12-31 12:15:23,559 - BERTopic - Embedding - Completed ✓
2024-12-31 12:15:23,559 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2024-12-31 12:15:39,086 - BERTopic - Dimensionality - Completed ✓
2024-12-31 12:15:39,087 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parall

    Topic  Count                                Name  \
0      -1   8046  -1_20102020_年_占_比_中国_黄金_20002010_年   
1       0   3155                      0_美国_中国_赖清德_国家   
2       1   2736                1_台方_大陆_方面_台当局_台湾_方面   
3       2   2156              2_金门_金门_海域_厦金_海域_金门_渔船   
4       3   1701               3_海巡_台湾_海巡_台海_巡署_海巡_署   
5       4    315    4_决心_心里_比 谁_都 清楚_谁_都 清楚_心里_比 谁_都   
6       5    282               5_台海_地区_台湾地区_地区_和平_稳定   
7       6    280                 6_水域_禁止_水域_渔船_大陆_渔船   
8       7    199                 7_2016_年_个_月_4_个_全年   
9       8    176          8_表态_落到实处_停止_煽风点火_火力_支持_台独   
10      9    173     9_公司_邓华_丰富_商品房 租赁_房_租赁_中介 公司_长租   
11     10    164                10_不幸_不幸遇难_2_人_不幸_罹难   
12     11    154               11_诗词_王_进兵_点击_下图_年_新年   
13     12    126   12_文章_正_能量 无_低俗 不良_引导_过程_图片_不良_引导   

                                       Representation  \
0   [20102020_年, 占_比, 中国_黄金, 20002010_年, 流出_地区, 地区...   
1   [美国, 中国, 赖清德, 国家, 台湾, 政治, 民进党_当局, 国台办, 国际

In [8]:
import pandas as pd
from bertopic import BERTopic
import numpy as np

# Step 1: 載入已保存的模型
model_save_path = "./model/bertopic_seed_words_2D_newStopWords"
topic_model = BERTopic.load(model_save_path)

# Step 2: 讀取原始數據
input_df = pd.read_csv('/Users/shuyuhsu/code_workspace/Xiemen_wechat_BERTTopic/data/kinmen_20241205_split_by_period copy.csv')

# 確保 content 欄位存在
if 'content' not in input_df.columns:
    raise ValueError("資料表中缺少 'content' 欄位！")

# Step 3: 使用 BERTopic 模型進行主題分類
topics, probs = topic_model.transform(input_df['content'])

# Step 4: 將主題編號加入原始數據表格
input_df['topic'] = topics

# 方法 1: 只保存主題的最大概率
input_df['topic_probability'] = np.max(probs, axis=1)

# 如果需要保存所有主題的概率，請使用方法 2
# 方法 2: 將主題概率展平為多列
# probs_df = pd.DataFrame(probs, columns=[f"topic_prob_{i}" for i in range(probs.shape[1])])
# input_df = pd.concat([input_df, probs_df], axis=1)

# Step 5: 儲存結果
output_path = './data/input_data_with_topics_newStopwords.csv'
input_df.to_csv(output_path, index=False)
print(f"已將分類結果儲存至: {output_path}")


Batches: 100%|██████████| 615/615 [00:25<00:00, 24.41it/s]
2024-12-31 12:16:28,949 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2024-12-31 12:16:31,533 - BERTopic - Dimensionality - Completed ✓
2024-12-31 12:16:31,534 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2024-12-31 12:16:32,200 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2024-12-31 12:16:33,562 - BERTopic - Probabilities - Completed ✓
2024-12-31 12:16:33,563 - BERTopic - Cluster - Completed ✓


已將分類結果儲存至: ./data/input_data_with_topics_newStopwords.csv


### 模型評估
1. 每個主題內的文章語意相似性高 <br>
2. 不同主題之間的主題差異性高

In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
# === 評估指標 ===

# 1. 每個主題內的文章相似性高
def calculate_within_topic_similarity(topic_model, embeddings, topics):
    topic_similarities = {}
    for topic in set(topics):
        if topic == -1:  # 忽略未分類的文檔
            continue
        # 提取屬於該主題的文檔嵌入向量
        topic_indices = [i for i, t in enumerate(topics) if t == topic]
        topic_embeddings = embeddings[topic_indices]
        # 計算嵌入向量的餘弦相似性
        if len(topic_embeddings) > 1:
            similarity_matrix = cosine_similarity(topic_embeddings)
            avg_similarity = np.mean(similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)])
            topic_similarities[topic] = avg_similarity
    return topic_similarities

# 計算每個主題內的平均相似性
within_topic_similarities = calculate_within_topic_similarity(topic_model, embeddings, topics)
print("\n每個主題內的平均相似性：")
for topic, similarity in within_topic_similarities.items():
    print(f"主題 {topic}: 平均相似性 = {similarity:.4f}")

# 2. 每個類別之間的差異性顯著
def calculate_between_topic_distances(topic_model, embeddings, topics):
    topic_centroids = {}
    for topic in set(topics):
        if topic == -1:  # 忽略未分類的文檔
            continue
        # 提取屬於該主題的文檔嵌入向量
        topic_indices = [i for i, t in enumerate(topics) if t == topic]
        topic_embeddings = embeddings[topic_indices]
        # 計算嵌入向量的中心
        topic_centroids[topic] = np.mean(topic_embeddings, axis=0)
    
    # 計算所有主題之間的距離
    topic_distances = {}
    topics_list = list(topic_centroids.keys())
    for i, topic_i in enumerate(topics_list):
        for j, topic_j in enumerate(topics_list):
            if i >= j:
                continue
            distance = np.linalg.norm(topic_centroids[topic_i] - topic_centroids[topic_j])
            topic_distances[(topic_i, topic_j)] = distance
    return topic_distances

# 計算主題之間的距離
between_topic_distances = calculate_between_topic_distances(topic_model, embeddings, topics)
print("\n主題之間的距離：")
for (topic_i, topic_j), distance in between_topic_distances.items():
    print(f"主題 {topic_i} 和主題 {topic_j}: 距離 = {distance:.4f}")


每個主題內的平均相似性：
主題 0: 平均相似性 = 0.5307
主題 1: 平均相似性 = 0.6594
主題 2: 平均相似性 = 0.6561
主題 3: 平均相似性 = 0.7052
主題 4: 平均相似性 = 0.7375
主題 5: 平均相似性 = 0.7513

主題之間的距離：
主題 0 和主題 1: 距離 = 0.4593
主題 0 和主題 2: 距離 = 0.3986
主題 0 和主題 3: 距離 = 0.4013
主題 0 和主題 4: 距離 = 0.5507
主題 0 和主題 5: 距離 = 0.5867
主題 1 和主題 2: 距離 = 0.5610
主題 1 和主題 3: 距離 = 0.5873
主題 1 和主題 4: 距離 = 0.7465
主題 1 和主題 5: 距離 = 0.6933
主題 2 和主題 3: 距離 = 0.5399
主題 2 和主題 4: 距離 = 0.5839
主題 2 和主題 5: 距離 = 0.5605
主題 3 和主題 4: 距離 = 0.7186
主題 3 和主題 5: 距離 = 0.6832
主題 4 和主題 5: 距離 = 0.7545


##### 模型UMAP and HBDSCAN 的分類視覺化

In [8]:
print(umap_embeddings.shape)  # 應該是 (n_samples, n_components)


NameError: name 'umap_embeddings' is not defined

In [9]:
import pandas as pd
# embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # 輕量級嵌入模型
# embeddings = embedding_model.encode(texts)
# 保存降維後的向量和聚類標籤
umap_embeddings = topic_model.umap_model.transform(embeddings)
hdbscan_labels = topic_model.hdbscan_model.labels_

# 提取每個文本的最大主題概率
probs_max = [max(prob) if prob is not None else None for prob in probs]

# 確保 texts 和 topics 的長度一致
assert len(texts) == len(topics) == len(probs), "數據長度不一致！"

# 提取 UMAP 的前兩維
umap_dim1 = umap_embeddings[:, 0]  # 第一維
umap_dim2 = umap_embeddings[:, 1]  # 第二維

# 創建 DataFrame
results = pd.DataFrame({
    "text": texts,               # 文本
    "topic": topics,             # 主題
    "probability": probs_max,    # 最大主題概率
    "umap_dim1": umap_dim1,      # UMAP 第一維
    "umap_dim2": umap_dim2       # UMAP 第二維
})

# 保存到 CSV
results.to_csv("./results/bertopic_results.csv", index=False, encoding="utf-8")
print("降維與聚類結果已保存至 ./results/bertopic_results.csv")


#==================主題關鍵詞輸出＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝
# 提取主題關鍵詞
topic_keywords = topic_model.get_topics()

# 將主題關鍵詞轉換為 DataFrame
keywords_list = []
for topic_id, keywords in topic_keywords.items():
    for word, score in keywords:
        keywords_list.append({"Topic": topic_id, "Word": word, "Score": score})

keywords_df = pd.DataFrame(keywords_list)

# 保存主題關鍵詞
keywords_df.to_csv("./results/topic_keywords.csv", index=False, encoding="utf-8")
print("主題關鍵詞已保存至 ./results/topic_keywords.csv")

#============當前BERTopic模型參數儲存==========================
import json

# 保存模型參數
model_config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "umap": {
        "n_neighbors": 10,
        "n_components": 2,
        "metric": "cosine"
    },
    "hdbscan": {
        "min_cluster_size": 100,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "prediction_data": True,
        "cluster_selection_epsilon": 0.1
    },
    "vectorizer": {
        "ngram_range": (1, 3),
        "stop_words": stop_words,
        "max_features": 10000
    },
    "ctfidf": {
        "seed_words": ["兩岸", "漁權", "爭議", "海巡"],
        "bm25_weighting": True,
        "reduce_frequent_words": True
    },
    "bertopic": {
        "top_n_words": 15,
        "min_topic_size": 100,
        "nr_topics": "auto"
    }
}

# 保存到 JSON 文件
with open("./results/model_config.json", "w", encoding="utf-8") as f:
    json.dump(model_config, f, ensure_ascii=False, indent=4)
print("模型參數已保存至 ./results/model_config.json")




降維與聚類結果已保存至 ./results/bertopic_results.csv
主題關鍵詞已保存至 ./results/topic_keywords.csv
模型參數已保存至 ./results/model_config.json


##### 模型可視化

In [10]:
import os
import webbrowser

# 定義保存目錄
output_dir = "./visualization_results_html"

# 檢查資料夾是否存在，若不存在則創建
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 保存並自動打開主題分佈圖
fig_topics = topic_model.visualize_topics()
topics_path = os.path.join(output_dir, "topics_visualization.html")
fig_topics.write_html(topics_path)
webbrowser.open(topics_path)
print(f"主題分佈圖已保存至 {topics_path}，並已在瀏覽器中打開。")

# 保存並自動打開主題關鍵詞柱狀圖
fig_barchart = topic_model.visualize_barchart(top_n_topics=10)
barchart_path = os.path.join(output_dir, "barchart_visualization.html")
fig_barchart.write_html(barchart_path)
webbrowser.open(barchart_path)
print(f"主題關鍵詞柱狀圖已保存至 {barchart_path}，並已在瀏覽器中打開。")

# 保存並自動打開主題間距離圖
fig_hierarchy = topic_model.visualize_hierarchy()
hierarchy_path = os.path.join(output_dir, "hierarchy_visualization.html")
fig_hierarchy.write_html(hierarchy_path)
webbrowser.open(hierarchy_path)
print(f"主題間距離圖已保存至 {hierarchy_path}，並已在瀏覽器中打開。")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


主題分佈圖已保存至 ./visualization_results_html/topics_visualization.html，並已在瀏覽器中打開。


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


主題關鍵詞柱狀圖已保存至 ./visualization_results_html/barchart_visualization.html，並已在瀏覽器中打開。
主題間距離圖已保存至 ./visualization_results_html/hierarchy_visualization.html，並已在瀏覽器中打開。


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


---

In [13]:
import plotly.io as pio
pio.renderers.default = "browser"

# 可視化主題分佈
topic_model.visualize_barchart(top_n_topics=5).show()

# 保存主題模型
topic_model.save("bertopic_model")

# 將主題與概率添加到數據框中
df["topic"] = topics
df["probability"] = probs

# 輸出結果
print(df[["_id", "topic", "probability"]])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2024-12-18 17:31:16,027 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


ValueError: Expected a 1D array, got an array with shape (19663, 4)

### loading the past bertopic model

In [1]:
from bertopic import BERTopic

# 載入已保存的模型
topic_model = BERTopic.load("./bertopic_model")


/Users/shuyuhsu/miniconda3/envs/NLP3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
topic_model.get_document_info()

TypeError: BERTopic.get_document_info() missing 1 required positional argument: 'docs'